In [0]:
file_path = "/Volumes/workspace/default/landing/schools_02_12.csv"

df = spark.read.csv(file_path, header=True, inferSchema=True)
df.show(5)


In [0]:
spark.sql("SELECT current_catalog(), current_schema()").show(truncate=False)


In [0]:
spark.sql("USE CATALOG workspace")
spark.sql("USE SCHEMA default")


In [0]:
display(dbutils.fs.ls("/Volumes/workspace/default/landing/"))


In [0]:
file_path = "/Volumes/workspace/default/landing/schools_02_12.csv"

df = spark.read.csv(file_path, header=True, inferSchema=True)
df.show(5)


In [0]:
df.write.format("delta").mode("overwrite").saveAsTable("workspace.default.schools")


In [0]:
spark.sql("SHOW TABLES IN workspace.default").show(truncate=False)


In [0]:
spark.table("workspace.default.schools").show(5)


In [0]:
# ============================================================
# Schools DATA PROFILING (ONE-SHOT FULL NOTEBOOK CELL)
# Works in Databricks with Spark DataFrame `df`
# Fixes dotted column names like coordinates.latitude -> coordinates_latitude
# ============================================================

from pyspark.sql import functions as F
import pandas as pd

print("="*70)
print("🍽️  SCHOOLS DATA PROFILING")
print("="*70)


try:
    _ = df.columns
    print("✓ DataFrame `df` found.")
except NameError:
    raise NameError("❌ DataFrame `df` is not defined. Load your CSV into `df` first.")

print("\n--- Original Column Names (first 30) ---")
print(df.columns[:30])

# ------------------------------------------------------------
# STEP 2: Clean/Normalize Column Names
# Replace '.' and spaces and special chars with '_'
# Avoids UNRESOLVED_COLUMN issues like coordinates.latitude
# ------------------------------------------------------------
def normalize_col(c: str) -> str:
    c2 = c.strip()
    for ch in [".", " ", "-", "/", "(", ")", "[", "]", "{", "}", ":", ";", ",", "|"]:
        c2 = c2.replace(ch, "_")
    while "__" in c2:
        c2 = c2.replace("__", "_")
    return c2.strip("_")

old_cols = df.columns
new_cols = [normalize_col(c) for c in old_cols]

# Handle duplicates after normalization by appending _2, _3...
seen = {}
final_cols = []
for c in new_cols:
    if c not in seen:
        seen[c] = 1
        final_cols.append(c)
    else:
        seen[c] += 1
        final_cols.append(f"{c}_{seen[c]}")

# Apply renames
for o, n in zip(old_cols, final_cols):
    if o != n:
        df = df.withColumnRenamed(o, n)

print("\n✓ Column names normalized.")
print("\n--- Normalized Column Names (first 30) ---")
print(df.columns[:30])

# ------------------------------------------------------------
# STEP 3: Preview
# ------------------------------------------------------------
print("\n--- First 10 Rows ---")
df.show(10, truncate=False)

# ------------------------------------------------------------
# STEP 4: Basic Overview
# ------------------------------------------------------------
print("\n" + "="*70)
print("📋 DATASET OVERVIEW")
print("="*70)

row_count = df.count()
col_count = len(df.columns)

print(f"Total Rows: {row_count:,}")
print(f"Total Columns: {col_count}")

print("\n--- Schema ---")
df.printSchema()

print("\n--- Column Names ---")
for i, col_name in enumerate(df.columns, 1):
    print(f"{i:2}. {col_name}")

# ------------------------------------------------------------
# STEP 5: Detailed Column Analysis
# Type, Nulls, Distinct, Samples
# ------------------------------------------------------------
print("\n" + "="*70)
print("📊 DETAILED COLUMN ANALYSIS")
print("="*70)

summary_data = []

for col_name in df.columns:
    print(f"\n--- {col_name} ---")

    col_type = str(df.schema[col_name].dataType)
    print(f"Type: {col_type}")

    null_count = df.filter(F.col(col_name).isNull()).count()
    null_pct = (null_count / row_count * 100) if row_count > 0 else 0.0
    print(f"Nulls: {null_count:,} ({null_pct:.2f}%)")

    distinct_count = df.select(F.col(col_name)).distinct().count()
    print(f"Distinct: {distinct_count:,}")

    try:
        samples = (
            df.select(F.col(col_name))
              .where(F.col(col_name).isNotNull())
              .limit(3)
              .collect()
        )
        sample_values = [str(r[0])[:80] for r in samples]
        print(f"Samples: {sample_values}")
    except Exception:
        print("Samples: (could not retrieve)")

    summary_data.append({
        "Column": col_name,
        "Type": col_type.replace("Type", ""),
        "Nulls": int(null_count),
        "Null_Pct": round(float(null_pct), 2),
        "Distinct": int(distinct_count)
    })

# ------------------------------------------------------------
# STEP 6: Summary Table (Pandas)
# ------------------------------------------------------------
print("\n" + "="*70)
print("📋 COLUMN SUMMARY TABLE")
print("="*70)

summary_df = pd.DataFrame(summary_data)
try:
    display(summary_df)  # Databricks display
except Exception:
    print(summary_df.to_string(index=False))

# ------------------------------------------------------------
# STEP 7: Numeric Statistics
# ------------------------------------------------------------
print("\n" + "="*70)
print("🔢 NUMERIC COLUMN STATISTICS")
print("="*70)

from pyspark.sql.types import *

numeric_cols = [
    f.name for f in df.schema.fields
    if isinstance(f.dataType, (DoubleType, IntegerType, LongType, FloatType, DecimalType, ShortType))
]


print(f"Numeric Columns Found: {len(numeric_cols)}")
if numeric_cols:
    print("Columns:", ", ".join(numeric_cols))
    df.select(numeric_cols).describe().show(truncate=False)
else:
    print("No numeric columns found.")

# ------------------------------------------------------------
# STEP 8: Missing Data Report (sorted)
# ------------------------------------------------------------
print("\n" + "="*70)
print("❌ MISSING DATA REPORT")
print("="*70)

missing_sorted = sorted(summary_data, key=lambda x: x["Null_Pct"], reverse=True)

print(f"\n{'Column':<35} {'Missing':<15} {'Percent':<10}")
print("-" * 65)

cols_with_missing = 0
for info in missing_sorted:
    if info["Null_Pct"] > 0:
        cols_with_missing += 1
        print(f"{info['Column']:<35} {info['Nulls']:<15,} {info['Null_Pct']:<10.2f}")

print(f"\n✓ {cols_with_missing} columns have missing data")

# ------------------------------------------------------------
# STEP 9: Duplicate Analysis
# ------------------------------------------------------------
print("\n" + "="*70)
print("🔄 DUPLICATE ANALYSIS")
print("="*70)

distinct_rows = df.distinct().count()
duplicate_rows = row_count - distinct_rows
duplicate_pct = (duplicate_rows / row_count * 100) if row_count > 0 else 0.0

print(f"Total Rows:     {row_count:>12,}")
print(f"Distinct Rows:  {distinct_rows:>12,}")
print(f"Duplicates:     {duplicate_rows:>12,} ({duplicate_pct:.2f}%)")

# ------------------------------------------------------------
# STEP 10: Categorical Frequencies (first 3 suitable columns)
# Criteria: 2..100 distinct values
# ------------------------------------------------------------
print("\n" + "="*70)
print("📊 CATEGORICAL COLUMN FREQUENCIES")
print("="*70)

categorical_cols = []
for col_name in df.columns:
    if col_name not in numeric_cols:
        try:
            dcount = df.select(F.col(col_name)).distinct().count()
            if 1 < dcount <= 100:
                categorical_cols.append((col_name, dcount))
        except Exception:
            pass

print(f"Found {len(categorical_cols)} categorical columns (2–100 unique values).")
for col_name, dcount in categorical_cols[:3]:
    print(f"\n--- {col_name} (distinct: {dcount}) ---")
    df.groupBy(F.col(col_name)).count().orderBy(F.desc("count")).show(10, truncate=False)

# ------------------------------------------------------------
# FINAL SUMMARY
# ------------------------------------------------------------
print("\n" + "="*70)
print("✅ PROFILING COMPLETE!")
print("="*70)

total_cells = row_count * col_count
null_cells = sum(x["Nulls"] for x in summary_data)
complete_cells = total_cells - null_cells
completeness = (complete_cells / total_cells * 100) if total_cells > 0 else 0.0

print(f"\n📊 OVERALL SUMMARY:")
print(f"  • Total Rows:              {row_count:>12,}")
print(f"  • Total Columns:           {col_count:>12,}")
print(f"  • Total Data Points:       {total_cells:>12,}")
print(f"  • Complete Data Points:    {complete_cells:>12,}")
print(f"  • Missing Data Points:     {null_cells:>12,}")
print(f"  • Data Completeness:       {completeness:>11.2f}%")

print(f"\n📈 COLUMN BREAKDOWN:")
print(f"  • Numeric Columns:         {len(numeric_cols):>12}")
print(f"  • Categorical Columns:     {len(categorical_cols):>12}")
print(f"  • Columns with Missing:    {cols_with_missing:>12}")

print(f"\n🔍 DATA QUALITY:")
print(f"  • Duplicate Rows:          {duplicate_rows:>12,} ({duplicate_pct:.2f}%)")
print(f"  • Unique Rows:             {distinct_rows:>12,}")

print("\n" + "="*70)
print("🎉 Done. Review outputs above.")
print("="*70)
